# Unicycle vs Kinematic-bicycle PINN — Step 1 comparison

**Inputs:** your CARLA recording (`ground_truth.txt`, `vehicle_sensors.txt`) + your trained **unicycle** weights.
**This notebook:** (1) loads the unicycle, (2) trains the new **kinematic-bicycle** PINN at the same settings,
(3) compares them on the CARLA GNSS-outage window.

The two models are identical except for the motion model: the unicycle moves along the heading `psi`; the
kinematic bicycle moves along `psi + beta`, with slip `beta = asin(L_R * wz / v)` from the gyro + speed
(IMU-only, no steering). `L_R -> 0` recovers the unicycle exactly.

> **Fair-comparison note:** train the kinematic at the **same `N_TRAIN`** your unicycle weights used. If the
> saved unicycle is a different size (your file looks like n=500), either set `N_TRAIN` to match, or set
> `RETRAIN_UNICYCLE = True` so both models are trained identically here — strictly apples-to-apples.

**Run All** after setting the paths and `L_R`.

In [ ]:
# ============================ CONFIG -- edit these ============================
# INPUTS: your CARLA recording (the IMU is derived from these) + your trained UNICYCLE weights.
GROUND_TRUTH    = "/kaggle/input/datasets/aminemoussi1/carla-sensors/ground_truth.txt"
VEHICLE_SENSORS = "/kaggle/input/datasets/aminemoussi1/carla-sensors/vehicle_sensors.txt"

UNICYCLE_PATH   = "/kaggle/input/models/aminemoussi1/pinn/pytorch/default/1/pinn_n500_closed_form.pt"  # weights to LOAD
KINEMATIC_PATH  = "/kaggle/working/pinn_kinematic.pt"   # kinematic weights (trained here -> /kaggle/working)

OUTAGE  = (4.25, 33.0)    # GNSS-outage window (s)
DT_PHYS = 0.00825         # CARLA tick (121 Hz)
DT      = 0.1             # PINN rate (10 Hz)

# --- kinematic-bicycle integrator ---
# velocity points along (psi + beta), beta = asin(L_R * wz / v).  L_R -> 0 == unicycle.
L_R = 1.5                 # rear-axle -> CoG distance (m); ~half the wheelbase. SET to your CARLA vehicle.

# --- training (MATCH the settings your unicycle weights were trained at, for a fair compare) ---
N_TRAIN, N_VAL, N_EPOCHS = 600, 100, 600
LAMBDA_BIAS = 100_000.0
RETRAIN_UNICYCLE  = False  # False = load UNICYCLE_PATH.  True = retrain unicycle here at the settings above
                           #         (use True if your saved weights are a different N_TRAIN, e.g. n=500)
RETRAIN_KINEMATIC = True   # train the kinematic fresh (False = reuse a saved KINEMATIC_PATH)

N_EVAL_SEEDS = 20          # noise realizations averaged over for the CARLA comparison
WILL_PEAK    = 2.0         # Will's fused-system peak, reference line
# ============================================================================
print("config OK | L_R = %.2f m | N_TRAIN = %d | lambda_bias = %g" % (L_R, N_TRAIN, LAMBDA_BIAS))

In [ ]:
import os, numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)
for name, p in [("ground_truth", GROUND_TRUTH), ("vehicle_sensors", VEHICLE_SENSORS), ("unicycle weights", UNICYCLE_PATH)]:
    print(("OK    " if os.path.exists(p) else "MISS  ") + name + ": " + p)

## Building blocks (your validated v7 code + the kinematic branch)

In [ ]:
def _prim_straight(n_steps):
    return np.zeros(n_steps), np.zeros(n_steps)

def _prim_arc(n_steps, v_in, radius, direction):
    # Constant-radius turn at the entering speed; ax=0 keeps v constant.
    wz_val = direction * v_in / radius
    return np.zeros(n_steps), np.full(n_steps, wz_val)

def _prim_speed_change(n_steps, target_ax):
    return np.full(n_steps, target_ax), np.zeros(n_steps)


def compose_window(primitives_spec, v0, yaw0, dt=0.1, duration=30.0, integrator='euler'):
    """integrator: 'euler' or 'closed_form'. Both produce same IMU, different x_gt/y_gt."""
    N = int(round(duration / dt)) + 1
    ax_arr = np.zeros(N)
    wz_arr = np.zeros(N)

    n_steps_per = [int(round(p[2] / dt)) for p in primitives_spec]
    n_steps_per[-1] += (N - 1) - sum(n_steps_per)

    v_now = v0
    idx = 0
    for (ptype, kwargs, _), n_steps in zip(primitives_spec, n_steps_per):
        if ptype == 'straight':
            ax_seg, wz_seg = _prim_straight(n_steps)
        elif ptype == 'arc':
            ax_seg, wz_seg = _prim_arc(n_steps, v_now, kwargs['radius'], kwargs['direction'])
        elif ptype == 'speed_change':
            ax_seg, wz_seg = _prim_speed_change(n_steps, kwargs['ax'])
            v_now = v_now + kwargs['ax'] * (n_steps * dt)
        else:
            raise ValueError(ptype)
        ax_arr[idx:idx + n_steps] = ax_seg
        wz_arr[idx:idx + n_steps] = wz_seg
        idx += n_steps
    ax_arr[-1] = ax_arr[-2]; wz_arr[-1] = wz_arr[-2]   # avoid trailing-zero cosmetic

    # Integrate forward using the requested scheme
    x = np.zeros(N); y = np.zeros(N); v = np.zeros(N); yaw = np.zeros(N)
    x[0], y[0], v[0], yaw[0] = 0.0, 0.0, v0, yaw0
    eps = 1e-6
    for i in range(N - 1):
        a, w   = ax_arr[i], wz_arr[i]
        v_i, psi = v[i], yaw[i]
        if integrator == 'euler':
            x[i+1] = x[i] + v_i * np.cos(psi) * dt
            y[i+1] = y[i] + v_i * np.sin(psi) * dt
        elif integrator == 'closed_form':
            if abs(w) < eps:
                x[i+1] = x[i] + v_i * np.cos(psi) * dt
                y[i+1] = y[i] + v_i * np.sin(psi) * dt
            else:
                x[i+1] = x[i] + (v_i / w) * (np.sin(psi + w*dt) - np.sin(psi))
                y[i+1] = y[i] - (v_i / w) * (np.cos(psi + w*dt) - np.cos(psi))
        elif integrator == 'rk4':
            def _d(xx, yy, vv, pp):
                return vv*np.cos(pp), vv*np.sin(pp), a, w
            k1 = _d(x[i], y[i], v_i, psi)
            k2 = _d(x[i]+0.5*dt*k1[0], y[i]+0.5*dt*k1[1], v_i+0.5*dt*k1[2], psi+0.5*dt*k1[3])
            k3 = _d(x[i]+0.5*dt*k2[0], y[i]+0.5*dt*k2[1], v_i+0.5*dt*k2[2], psi+0.5*dt*k2[3])
            k4 = _d(x[i]+dt*k3[0],     y[i]+dt*k3[1],     v_i+dt*k3[2],     psi+dt*k3[3])
            x[i+1] = x[i] + dt/6.0*(k1[0]+2*k2[0]+2*k3[0]+k4[0])
            y[i+1] = y[i] + dt/6.0*(k1[1]+2*k2[1]+2*k3[1]+k4[1])
        elif integrator == 'kinematic':
            # Kinematic bicycle (CoG): velocity along (psi + beta); beta from yaw-rate + speed (IMU-only).
            # Identical to closed_form except for the beta offset.  L_R -> 0 recovers the unicycle.
            if abs(w) < eps:
                x[i+1] = x[i] + v_i * np.cos(psi) * dt
                y[i+1] = y[i] + v_i * np.sin(psi) * dt
            else:
                beta = np.arcsin(np.clip(L_R * w / max(v_i, 1e-3), -0.99, 0.99))
                th   = psi + beta
                x[i+1] = x[i] + (v_i / w) * (np.sin(th + w*dt) - np.sin(th))
                y[i+1] = y[i] - (v_i / w) * (np.cos(th + w*dt) - np.cos(th))
        else:
            raise ValueError(integrator)
        yaw[i+1] = psi + w * dt
        v[i+1]   = v_i + a * dt
    ay_arr = v * wz_arr
    t = np.arange(0, duration + dt/2, dt)

    return {
        't': t, 'dt': np.full(N, dt),
        'x_gt': x, 'y_gt': y, 'v_gt': v, 'yaw_gt': yaw,
        'ax': ax_arr, 'ay': ay_arr, 'wz': wz_arr,
        'v0': float(v0), 'yaw0': float(yaw0),
        'drive': 'synthetic_composed',
        'gt_integrator': integrator,
    }


def sample_complexity_window(complexity, seed, integrator='euler'):
    """Same as before but propagates integrator to compose_window."""
    rng = np.random.default_rng(seed)
    v0   = float(rng.uniform(5.0, 20.0))
    yaw0 = float(rng.uniform(-np.pi, np.pi))
    if complexity == 'simple':
        n_prims      = int(rng.choice([1, 2]))
        prim_probs   = {'straight': 0.60, 'arc': 0.20, 'speed_change': 0.20}
        radius_range = (60.0, 150.0); ax_range = (-0.3, 0.3)
    else:
        n_prims      = int(rng.choice([3, 4, 5]))
        prim_probs   = {'straight': 0.30, 'arc': 0.40, 'speed_change': 0.30}
        radius_range = (20.0, 100.0); ax_range = (-1.0, 1.0)
    names = list(prim_probs.keys())
    probs = np.array(list(prim_probs.values())); probs /= probs.sum()
    raw = rng.uniform(0.7, 1.3, n_prims); raw *= 30.0 / raw.sum()
    durations = raw.tolist()
    while min(durations) < 3.0 and len(durations) > 1:
        i = int(np.argmin(durations))
        j = i + 1 if i + 1 < len(durations) else i - 1
        durations[j] += durations[i]; durations.pop(i)
    s = sum(durations); durations = [d * 30.0 / s for d in durations]
    spec = []
    for d in durations:
        ptype = str(rng.choice(names, p=probs))
        if ptype == 'arc':
            kw = {'radius': float(rng.uniform(*radius_range)),
                  'direction': float(rng.choice([-1.0, 1.0]))}
        elif ptype == 'speed_change':
            kw = {'ax': float(rng.uniform(*ax_range))}
        else:
            kw = {}
        spec.append((ptype, kw, d))
    w = compose_window(spec, v0=v0, yaw0=yaw0, integrator=integrator)
    w['spec']       = spec
    w['complexity'] = complexity
    return w, spec


def make_dataset_v2(n_windows, seed_base=0, p_simple=0.8, integrator='euler'):
    rng = np.random.default_rng(seed_base)
    windows = []
    for i in range(n_windows):
        complexity = 'simple' if rng.random() < p_simple else 'complex'
        w, spec = sample_complexity_window(complexity, seed=seed_base + i, integrator=integrator)
        windows.append(w)
    n_simp = sum(1 for w in windows if w['complexity'] == 'simple')
    print(f"make_dataset_v2(n={n_windows}, integrator={integrator!r}):  "
          f"simple={n_simp}, complex={len(windows)-n_simp}")
    return windows

In [ ]:
NOISE_PARAMS_C = dict(
    sigma_b0_a=0.10,
    sigma_b0_w=0.015,
    sigma_a=0.05,          # white noise on accel (m/s²)
    sigma_w=0.01,          # white noise on gyro  (rad/s)
    sigma_brwa=0.001,      # accel bias random walk (m/s²/√s)
    sigma_brww=0.0002,     # gyro bias random walk  (rad/s/√s)
)

def inject_full_noise(window, noise_params, seed):
    """Constant bias + white noise + bias random walk."""
    rng = np.random.default_rng(seed)
    N = len(window['t'])
    dt = float(window['dt'][0])
    b0_ax = rng.normal(0.0, noise_params['sigma_b0_a'])
    b0_wz = rng.normal(0.0, noise_params['sigma_b0_w'])
    white_ax = rng.normal(0.0, noise_params['sigma_a'], N)
    white_wz = rng.normal(0.0, noise_params['sigma_w'], N)
    brw_ax = np.cumsum(rng.normal(0.0, noise_params['sigma_brwa']*np.sqrt(dt), N))
    brw_wz = np.cumsum(rng.normal(0.0, noise_params['sigma_brww']*np.sqrt(dt), N))
    bias_a_profile = b0_ax + brw_ax
    bias_w_profile = b0_wz + brw_wz
    return {
        'ax': window['ax'] + bias_a_profile + white_ax,
        'ay': window['ay'],
        'wz': window['wz'] + bias_w_profile + white_wz,
        'bias_a_profile': bias_a_profile,
        'bias_w_profile': bias_w_profile,
    }


def stack_batch_full_noise_v2(windows, noise_params, base_seed):
    """Same as stack_batch_full_noise but also returns per-timestep bias profiles."""
    noisy = [inject_full_noise(w, noise_params, base_seed + i) for i, w in enumerate(windows)]
    ax = torch.tensor(np.stack([n['ax'] for n in noisy]), dtype=torch.float32, device=DEVICE)
    ay = torch.tensor(np.stack([n['ay'] for n in noisy]), dtype=torch.float32, device=DEVICE)
    wz = torch.tensor(np.stack([n['wz'] for n in noisy]), dtype=torch.float32, device=DEVICE)
    dt = torch.tensor(np.stack([w['dt'] for w in windows]), dtype=torch.float32, device=DEVICE)
    v0 = torch.tensor([w['v0']   for w in windows], dtype=torch.float32, device=DEVICE)
    yaw0 = torch.tensor([w['yaw0'] for w in windows], dtype=torch.float32, device=DEVICE)
    b_a_mean = torch.tensor([n['bias_a_profile'].mean() for n in noisy],
                            dtype=torch.float32, device=DEVICE)
    b_w_mean = torch.tensor([n['bias_w_profile'].mean() for n in noisy],
                            dtype=torch.float32, device=DEVICE)
    # NEW: per-timestep bias profile tensors (B, T)
    ba_profile = torch.tensor(np.stack([n['bias_a_profile'] for n in noisy]),
                              dtype=torch.float32, device=DEVICE)
    bw_profile = torch.tensor(np.stack([n['bias_w_profile'] for n in noisy]),
                              dtype=torch.float32, device=DEVICE)
    return ax, ay, wz, dt, v0, yaw0, b_a_mean, b_w_mean, ba_profile, bw_profile

In [ ]:
class BiasGRU(nn.Module):
    def __init__(self, hidden=32, n_layers=1):
        super().__init__()
        self.gru  = nn.GRU(input_size=6, hidden_size=hidden,
                           num_layers=n_layers, batch_first=True)
        self.head = nn.Linear(hidden, 2)
        # Small (NOT zero) weights so initial bias predictions are tiny
        # but gradients can flow back into the GRU.
        nn.init.normal_(self.head.weight, std=1e-3)
        nn.init.zeros_(self.head.bias)

    def forward(self, imu_seq, state_seq):
        x = torch.cat([imu_seq, state_seq], dim=-1)
        h, _ = self.gru(x)
        bias = self.head(h)
        return bias


class GreyBoxPINN(nn.Module):
    def __init__(self, hidden=32, integrator='euler'):
        super().__init__()
        self.bias_net = BiasGRU(hidden=hidden)
        self.integrator = integrator

    def forward(self, ax, ay, wz, dt, v0, yaw0):
        B, T = ax.shape
        v0_seq     = v0.unsqueeze(1).expand(B, T)
        sin_yaw0_s = torch.sin(yaw0).unsqueeze(1).expand(B, T)
        cos_yaw0_s = torch.cos(yaw0).unsqueeze(1).expand(B, T)
        imu_seq    = torch.stack([ax, ay, wz], dim=-1)
        state_seq  = torch.stack([v0_seq, sin_yaw0_s, cos_yaw0_s], dim=-1)
        bias_seq   = self.bias_net(imu_seq, state_seq)

        x_list   = [torch.zeros(B, device=ax.device)]
        y_list   = [torch.zeros(B, device=ax.device)]
        v_list   = [v0]
        yaw_list = [yaw0]
        eps = 1e-4

        for i in range(T - 1):
            b_a = bias_seq[:, i, 0]; b_w = bias_seq[:, i, 1]
            ax_clean = ax[:, i] - b_a
            wz_clean = wz[:, i] - b_w
            dt_i  = dt[:, i]
            v_cur = v_list[-1]; psi = yaw_list[-1]

            if self.integrator == 'euler':
                x_next = x_list[-1] + v_cur * torch.cos(psi) * dt_i
                y_next = y_list[-1] + v_cur * torch.sin(psi) * dt_i
            elif self.integrator == 'closed_form':
                psi_new = psi + wz_clean * dt_i
                mask    = torch.abs(wz_clean) < eps
                safe_w  = torch.where(mask, torch.full_like(wz_clean, eps), wz_clean)
                x_cf = x_list[-1] + v_cur * (torch.sin(psi_new) - torch.sin(psi)) / safe_w
                y_cf = y_list[-1] - v_cur * (torch.cos(psi_new) - torch.cos(psi)) / safe_w
                x_st = x_list[-1] + v_cur * torch.cos(psi) * dt_i
                y_st = y_list[-1] + v_cur * torch.sin(psi) * dt_i
                x_next = torch.where(mask, x_st, x_cf)
                y_next = torch.where(mask, y_st, y_cf)
            elif self.integrator == 'rk4':
                a = ax_clean; w = wz_clean
                def _d(xx, yy, vv, pp):
                    return vv*torch.cos(pp), vv*torch.sin(pp), a, w
                k1 = _d(x_list[-1], y_list[-1], v_cur, psi)
                k2 = _d(x_list[-1]+0.5*dt_i*k1[0], y_list[-1]+0.5*dt_i*k1[1], v_cur+0.5*dt_i*k1[2], psi+0.5*dt_i*k1[3])
                k3 = _d(x_list[-1]+0.5*dt_i*k2[0], y_list[-1]+0.5*dt_i*k2[1], v_cur+0.5*dt_i*k2[2], psi+0.5*dt_i*k2[3])
                k4 = _d(x_list[-1]+dt_i*k3[0],     y_list[-1]+dt_i*k3[1],     v_cur+dt_i*k3[2],     psi+dt_i*k3[3])
                x_next = x_list[-1] + dt_i/6.0*(k1[0]+2*k2[0]+2*k3[0]+k4[0])
                y_next = y_list[-1] + dt_i/6.0*(k1[1]+2*k2[1]+2*k3[1]+k4[1])
            elif self.integrator == 'kinematic':
                # Kinematic bicycle (CoG): velocity along (psi + beta); beta = asin(L_R*wz/v).
                # Same wz as the unicycle (IMU-only, no steering).  L_R -> 0 == unicycle.
                mask   = torch.abs(wz_clean) < eps
                safe_w = torch.where(mask, torch.full_like(wz_clean, eps), wz_clean)
                v_safe = torch.clamp(v_cur, min=1e-3)
                ratio  = torch.clamp(L_R * wz_clean / v_safe, -0.99, 0.99)
                beta   = torch.where(mask, torch.zeros_like(ratio), torch.asin(ratio))
                th     = psi + beta
                th_new = th + wz_clean * dt_i
                x_cf = x_list[-1] + v_cur * (torch.sin(th_new) - torch.sin(th)) / safe_w
                y_cf = y_list[-1] - v_cur * (torch.cos(th_new) - torch.cos(th)) / safe_w
                x_st = x_list[-1] + v_cur * torch.cos(psi) * dt_i
                y_st = y_list[-1] + v_cur * torch.sin(psi) * dt_i
                x_next = torch.where(mask, x_st, x_cf)
                y_next = torch.where(mask, y_st, y_cf)
            else:
                raise ValueError(self.integrator)

            v_next   = v_cur + ax_clean * dt_i
            yaw_next = psi   + wz_clean * dt_i
            x_list.append(x_next); y_list.append(y_next)
            v_list.append(v_next); yaw_list.append(yaw_next)

        x   = torch.stack(x_list,   dim=1)
        y   = torch.stack(y_list,   dim=1)
        v   = torch.stack(v_list,   dim=1)
        yaw = torch.stack(yaw_list, dim=1)
        return x, y, v, yaw, bias_seq

In [ ]:
def train_noisy_with_bias(model, train_windows, val_windows, noise_params,
                          stack_fn=None,
                          n_epochs=600, batch_size=32, lr=1e-3, log_every=20,
                          lambda_smooth=100.0, lambda_bias=0.0):
    """
    Trainer with optional direct bias supervision.
    lambda_bias = 0 -> same behaviour as train_noisy.
    lambda_bias > 0 -> add MSE(predicted_bias_profile, true_bias_profile) to the loss.
    stack_fn must return 10 values (see stack_batch_full_noise_v2).
    """
    if stack_fn is None:
        stack_fn = stack_batch_full_noise_v2

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    history = {'epoch': [], 'pos_loss': [], 'smooth_loss': [], 'bias_loss': [],
               'val_drift_30s': [], 'bias_corr_a': [], 'bias_corr_w': []}

    for ep in range(n_epochs):
        model.train()
        idxs = np.random.permutation(len(train_windows))
        epoch_seed = 1_000_000 + ep * 10_000
        last_pos, last_smooth, last_bias = 0.0, 0.0, 0.0

        for start in range(0, len(train_windows), batch_size):
            batch_idx = idxs[start:start + batch_size]
            batch = [train_windows[i] for i in batch_idx]
            ax_b, ay_b, wz_b, dt_b, v0_b, yaw0_b, _, _, ba_prof, bw_prof = stack_fn(
                batch, noise_params, epoch_seed + start)
            x_gt = torch.tensor(np.stack([w['x_gt'] for w in batch]),
                                dtype=torch.float32, device=DEVICE)
            y_gt = torch.tensor(np.stack([w['y_gt'] for w in batch]),
                                dtype=torch.float32, device=DEVICE)

            x_p, y_p, _, _, bias_seq = model(ax_b, ay_b, wz_b, dt_b, v0_b, yaw0_b)

            pos_loss    = ((x_p - x_gt)**2 + (y_p - y_gt)**2).mean()
            smooth_loss = ((bias_seq[:, 1:, :] - bias_seq[:, :-1, :])**2).mean()
            bias_loss   = ((bias_seq[:, :, 0] - ba_prof)**2 +
                           (bias_seq[:, :, 1] - bw_prof)**2).mean()

            loss = pos_loss + lambda_smooth * smooth_loss + lambda_bias * bias_loss

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            last_pos    = pos_loss.item()
            last_smooth = smooth_loss.item()
            last_bias   = bias_loss.item()
        scheduler.step()

        if ep % log_every == 0 or ep == n_epochs - 1:
            model.eval()
            with torch.no_grad():
                val_drifts, true_bas, pred_bas, true_bws, pred_bws = [], [], [], [], []
                for vw in val_windows:
                    for s in range(3):
                        ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v, b_a_t, b_w_t, _, _ = stack_fn(
                            [vw], noise_params, base_seed=900_000 + s * 100)
                        x_v, y_v, _, _, bias_v = model(ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v)
                        drift = float(torch.sqrt(
                            (x_v[0, -1] - vw['x_gt'][-1])**2 +
                            (y_v[0, -1] - vw['y_gt'][-1])**2))
                        val_drifts.append(drift)
                        true_bas.append(float(b_a_t[0]))
                        pred_bas.append(float(bias_v[:,:,0].mean()))
                        true_bws.append(float(b_w_t[0]))
                        pred_bws.append(float(bias_v[:,:,1].mean()))
                corr_a = np.corrcoef(true_bas, pred_bas)[0, 1]
                corr_w = np.corrcoef(true_bws, pred_bws)[0, 1]

            history['epoch'].append(ep)
            history['pos_loss'].append(last_pos)
            history['smooth_loss'].append(last_smooth)
            history['bias_loss'].append(last_bias)
            history['val_drift_30s'].append(float(np.mean(val_drifts)))
            history['bias_corr_a'].append(float(corr_a))
            history['bias_corr_w'].append(float(corr_w))
            print(f"  ep {ep:4d}  pos={last_pos:7.2f}  bias={last_bias:.5f}  "
                  f"val_drift={np.mean(val_drifts):6.2f}m  "
                  f"corr(b_a)={corr_a:+.3f}  corr(b_w)={corr_w:+.3f}  "
                  f"lr={opt.param_groups[0]['lr']:.5f}")
    return history

## Build the CARLA outage window from your recording

In [ ]:
import numpy as np

def parse_pairs(path, n):
    rows = []
    for line in open(path):
        p = [s.strip() for s in line.strip().split(",")]
        if len(p) < 2 * n:
            continue
        try:
            rows.append([float(p[2 * i + 1]) for i in range(n)])
        except ValueError:
            continue
    return np.array(rows)

def smooth(a, w=5):
    pad = w // 2
    return np.convolve(np.pad(a, pad, mode="edge"), np.ones(w) / w, mode="valid")

# derive clean IMU + speed/yaw + true path from the raw recording, all on a 10 Hz grid
veh = parse_pairs(VEHICLE_SENSORS, 3)            # steer, speed, yaw(deg)
gt  = parse_pairs(GROUND_TRUTH, 3)               # x, y, z
t_p  = np.arange(len(veh)) * DT_PHYS
t_gt = np.arange(len(gt)) * DT_PHYS

t   = np.arange(t_p[0], t_p[-1], DT)
spd = smooth(np.interp(t, t_p, veh[:, 1]))
yaw = smooth(np.interp(t, t_p, np.unwrap(np.deg2rad(veh[:, 2]))))
ax_c = np.gradient(spd, DT)
wz_c = np.gradient(yaw, DT)
ay_c = spd * wz_c
gx = np.interp(t, t_gt, gt[:, 0]); gy = np.interp(t, t_gt, gt[:, 1])

i0, i1 = int(np.searchsorted(t, OUTAGE[0])), int(np.searchsorted(t, OUTAGE[1]))
sl = slice(i0, i1 + 1)
t_o = t[sl]; N = len(t_o)
v0, yaw0 = float(spd[i0]), float(yaw[i0])
xg = gx[sl] - gx[i0]; yg = gy[sl] - gy[i0]       # true path relative to the dropout point

carla = dict(t=t_o, dt=np.full(N, DT), ax=ax_c[sl], ay=ay_c[sl], wz=wz_c[sl],
             v0=v0, yaw0=yaw0, x_gt=xg, y_gt=yg)
print("outage window: %d samples (%.1f s), v0=%.1f m/s, yaw0=%.0f deg"
      % (N, t_o[-1] - t_o[0], v0, np.degrees(yaw0)))


## Train: load the unicycle, train the kinematic (identical settings)

In [ ]:
# ---- unicycle: load your weights (or retrain to match the kinematic exactly) ----
uni_model = GreyBoxPINN(integrator='closed_form').to(DEVICE)
if RETRAIN_UNICYCLE or not os.path.exists(UNICYCLE_PATH):
    print("training unicycle (closed_form, N_TRAIN=%d) ..." % N_TRAIN)
    tr = make_dataset_v2(N_TRAIN, seed_base=0,       integrator='closed_form')
    va = make_dataset_v2(N_VAL,   seed_base=500_000, integrator='closed_form')
    _  = train_noisy_with_bias(uni_model, tr, va, NOISE_PARAMS_C, n_epochs=N_EPOCHS, lambda_bias=LAMBDA_BIAS)
    torch.save(uni_model.state_dict(), "/kaggle/working/pinn_unicycle.pt")
    print("saved retrained unicycle -> /kaggle/working/pinn_unicycle.pt")
else:
    uni_model.load_state_dict(torch.load(UNICYCLE_PATH, map_location=DEVICE))
    print("loaded unicycle weights:", UNICYCLE_PATH)
uni_model.eval()

# ---- kinematic: train fresh at the SAME settings, on its own self-consistent synthetic GT ----
kin_model = GreyBoxPINN(integrator='kinematic').to(DEVICE)
if (not RETRAIN_KINEMATIC) and os.path.exists(KINEMATIC_PATH):
    kin_model.load_state_dict(torch.load(KINEMATIC_PATH, map_location=DEVICE))
    print("loaded kinematic weights:", KINEMATIC_PATH)
else:
    print("training kinematic (bicycle, L_R=%.2f, N_TRAIN=%d) ..." % (L_R, N_TRAIN))
    trk = make_dataset_v2(N_TRAIN, seed_base=0,       integrator='kinematic')
    vak = make_dataset_v2(N_VAL,   seed_base=500_000, integrator='kinematic')
    hist_kin = train_noisy_with_bias(kin_model, trk, vak, NOISE_PARAMS_C, n_epochs=N_EPOCHS, lambda_bias=LAMBDA_BIAS)
    torch.save(kin_model.state_dict(), KINEMATIC_PATH)
    print("saved kinematic weights:", KINEMATIC_PATH)
kin_model.eval()
print("both models ready (L_R = %.2f m)" % L_R)

## Compare: unicycle vs kinematic on the outage

In [ ]:
# ---- compare on the CARLA outage: SAME noise seeds for both models ----
def _dr_closed_form(ax, wz, v0, yaw0, dt):
    N=len(ax); x=np.zeros(N); y=np.zeros(N); v=v0; psi=yaw0; eps=1e-6
    for i in range(N-1):
        w=wz[i]
        if abs(w)<eps:
            x[i+1]=x[i]+v*np.cos(psi)*dt; y[i+1]=y[i]+v*np.sin(psi)*dt
        else:
            x[i+1]=x[i]+(v/w)*(np.sin(psi+w*dt)-np.sin(psi))
            y[i+1]=y[i]-(v/w)*(np.cos(psi+w*dt)-np.cos(psi))
        psi+=w*dt; v+=ax[i]*dt
    return x,y

def _run(m, ax, ay, wz, v0, yaw0, dt):
    A=lambda a: torch.tensor(a[None,:],dtype=torch.float32,device=DEVICE)
    with torch.no_grad():
        xp,yp,_,_,_=m(A(ax),A(ay),A(wz),A(np.full(len(ax),dt)),
                      torch.tensor([v0],dtype=torch.float32,device=DEVICE),
                      torch.tensor([yaw0],dtype=torch.float32,device=DEVICE))
    return xp[0].cpu().numpy(), yp[0].cpu().numpy()

xg, yg = carla["x_gt"], carla["y_gt"]
uni_pk, kin_pk, naive_pk = [], [], []; rep=None
for s in range(N_EVAL_SEEDS):
    nz = inject_full_noise(carla, NOISE_PARAMS_C, seed=12345+s)     # SAME seed -> identical IMU for both
    xu,yu = _run(uni_model, nz["ax"], nz["ay"], nz["wz"], v0, yaw0, DT)
    xk,yk = _run(kin_model, nz["ax"], nz["ay"], nz["wz"], v0, yaw0, DT)
    xn,yn = _dr_closed_form(nz["ax"], nz["wz"], v0, yaw0, DT)
    uni_pk.append(float(np.hypot(xu-xg, yu-yg).max()))
    kin_pk.append(float(np.hypot(xk-xg, yk-yg).max()))
    naive_pk.append(float(np.hypot(xn-xg, yn-yg).max()))
    if s==0: rep=dict(xu=xu,yu=yu,xk=xk,yk=yk,xn=xn,yn=yn)
uni_pk,kin_pk,naive_pk = map(np.array,(uni_pk,kin_pk,naive_pk))

print("=== peak drift over the outage (%d seeds, identical noise for both) ===" % N_EVAL_SEEDS)
print("naive (no PINN) : %5.2f +/- %.2f m   (worst %.2f)" % (naive_pk.mean(),naive_pk.std(),naive_pk.max()))
print("unicycle  PINN  : %5.2f +/- %.2f m   (worst %.2f)" % (uni_pk.mean(),  uni_pk.std(),  uni_pk.max()))
print("kinematic PINN  : %5.2f +/- %.2f m   (worst %.2f)" % (kin_pk.mean(),  kin_pk.std(),  kin_pk.max()))
print("kinematic - unicycle : mean %+.2f m,  worst %+.2f m" % (kin_pk.mean()-uni_pk.mean(), kin_pk.max()-uni_pk.max()))

r=rep; tt=carla["t"]-carla["t"][0]
fig,axs=plt.subplots(1,2,figsize=(14,5.5))
a=axs[0]
a.plot(xg,yg,"k-",lw=2.5,label="true (GPS)")
a.plot(r["xu"],r["yu"],"-",color="#0072B2",lw=1.8,label="unicycle (peak %.2f m)"%uni_pk.mean())
a.plot(r["xk"],r["yk"],"-",color="#D55E00",lw=1.8,label="kinematic (peak %.2f m)"%kin_pk.mean())
a.plot(r["xn"],r["yn"],"r--",lw=1.2,alpha=.7,label="naive (peak %.2f m)"%naive_pk.mean())
a.scatter([0],[0],c="b",s=60,zorder=5,label="GNSS drops")
a.set_aspect("equal"); a.legend(); a.set_xlabel("x (m)"); a.set_ylabel("y (m)")
a.set_title("Trajectory during outage - unicycle vs kinematic")
a=axs[1]
a.plot(tt,np.hypot(r["xu"]-xg,r["yu"]-yg),"-",color="#0072B2",label="unicycle")
a.plot(tt,np.hypot(r["xk"]-xg,r["yk"]-yg),"-",color="#D55E00",label="kinematic")
a.plot(tt,np.hypot(r["xn"]-xg,r["yn"]-yg),"r--",alpha=.7,label="naive")
a.legend(); a.set_xlabel("time into outage (s)"); a.set_ylabel("position error (m)")
a.set_title("Drift growth - unicycle vs kinematic")
plt.tight_layout(); plt.savefig("/kaggle/working/unicycle_vs_kinematic.png",dpi=120); plt.show()
print("saved -> /kaggle/working/unicycle_vs_kinematic.png")